In [105]:
2+2

4

In [106]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re


In [107]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [108]:
result = pd.read_parquet("results/results_myvideos_174/logits_output_meta.parquet")
print(result.shape)
del result['top5_classes']
result.head(4)

(150, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.70513,"[49, 106, 1, 145, 5]",True,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.77079,"[49, 106, 1, 145, 5]",True,True
2,260922-134214,cam2,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.89020,"[49, 1, 106, 96, 50]",True,True
3,260922-134301,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.75190,"[49, 106, 1, 5, 145]",True,True


In [109]:
d = np.load("results/results_myvideos_174/logits_output.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(150, 174)


NpzFile 'results/results_myvideos_174/logits_output.npz' with keys: logits, probs, labels, video_ids, cams...

# main result

In [110]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'total_evaluated': 150,
 'top1_accuracy': np.float64(0.486667),
 'top1_balanced_accuracy': np.float64(0.486667),
 'top5_accuracy': np.float64(0.9),
 'top5_balanced_accuracy': np.float64(0.9)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [111]:
result.groupby("cam")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
)

,top1_acc,top5_acc
cam,,
cam0,0.40000,0.86000
cam1,0.62000,0.98000
cam2,0.44000,0.86000


In [112]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.40000,0.86667
Moving [something] up,0.06667,1.00000
Piling [something] up,0.00000,1.00000
Plugging [something] into [something],1.00000,1.00000
Pushing [something] with [something],1.00000,1.00000
Putting [something] next to [something],0.40000,0.93333
Putting [something] on a surface,0.46667,1.00000
Putting [something] onto [something],0.26667,0.93333
Showing a photo of [something] to the camera,0.26667,0.26667


In [113]:
result[result.cam=='cam1'].groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.40000,1.00000
Moving [something] up,0.20000,1.00000
Piling [something] up,0.00000,1.00000
Plugging [something] into [something],1.00000,1.00000
Pushing [something] with [something],1.00000,1.00000
Putting [something] next to [something],0.60000,1.00000
Putting [something] on a surface,0.60000,1.00000
Putting [something] onto [something],0.60000,1.00000
Showing a photo of [something] to the camera,0.80000,0.80000


# Granular analysis

##### Distractor objects -> expected spike in action/verb -- P(Action|Noun) estimate
- Plug -> Plugging [something] into [something] -- 55%
- lady/person -> Showing a photo of [something] to the camera -- 96%
- magnet -> Attaching [something] to [something] -- 60%
- Books -> Piling [something] up -- 55%
- (~) Paper -> Tearing [something] into two pieces -- 25%

##### id2label

In [114]:
with open('./data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [115]:
result.head(1)

,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,Plugging [something] into [something],0.70513,"[49, 106, 1, 145, 5]",True,True


In [129]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','top1_conf','correct_top1']]
print(df.shape)
df.head(2)

(150, 8)


,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.705132,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.770786,True


In [130]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,prob_0,prob_1,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.705132,True,0.000001,0.025684,...,0.000006,0.000001,0.000004,0.000002,0.000002,0.000011,0.000093,0.000000,0.000000,0.000001
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.770786,True,0.000005,0.071694,...,0.000020,0.000005,0.000009,0.000004,0.000006,0.000010,0.000280,0.000000,0.000000,0.000004


#### Main Analysis 
Comparison of:
- P(label=3|session=plug_1) -- should be high.
- P(label=3|session=plug_2) -- if its high it means plug as distractor is increasing probabilty of label=3.
- P(label=3|session not like 'plug%')

In [131]:
distractor = 'plug'
label = 3

prob_col = f'prob_{label}'

df['session_temp'] = np.where(df['session'] == f'{distractor}_1', "Normal",
                     np.where(df['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
              
df.groupby(['session','session_temp']).agg(
    videos=('video_id', 'count'),
).reset_index()


,session,session_temp,videos
0,books_1,Baseline,15
1,books_2,Baseline,15
2,lady_1,Baseline,15
3,lady_2,Baseline,15
4,magnet_1,Baseline,15
5,magnet_2,Baseline,15
6,paper_1,Baseline,15
7,paper_2,Baseline,15
8,plug_1,Normal,15
9,plug_2,distractor_plug,15


In [132]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

In [135]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("=" * 20, 'Videos with distractor', "=" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")
            
        print("-" * 50)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.00181   0.00465  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.64753   0.16921  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.00002   0.00002  0.80000  260922-141236, 260922-141249, 260922-141308
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114 prob_107 prob_42  prob_6 prob_105
16  260922-141236  cam1  plug_2  Putting [something] next to [something]   107       114   0.70823        False  distractor_plug  0.70823  0.15316 0.02349 0.01718  0.01425

Top 5 Probability Breakdown:
  prob_114 ->  0.70823  |  P

In [136]:
avg_prob_in_diff_cases(df, distractor='lady', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady =========================
label 124: Showing a photo of [something] to the camera
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.00015   0.00056  0.60000  260922-134214, 260922-134301, 260922-134333
1           Normal       5    0.49422   0.26897  0.80000  260922-150726, 260922-150740, 260922-150754
2  distractor_lady       5    0.00061   0.00121  0.60000  260922-150856, 260922-150915, 260922-150927
==================== Videos with distractor ====================
          video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_107 prob_37 prob_42 prob_114  prob_6
106  260922-150856  cam1  lady_2  Putting [something] next to [something]   107       107   0.44613         True  distractor_lady  0.44613 0.25420 0.09648  0.08785 0.02439

Top 5 Probability Breakdown:
  prob_107 ->  0.446

In [137]:
avg_prob_in_diff_cases(df, distractor='magnet', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
        session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0           Baseline      40    0.01236   0.03419  0.65000  260922-134214, 260922-134301, 260922-134333
1             Normal       5    0.36716   0.40383  0.40000  260922-143234, 260922-143255, 260922-143323
2  distractor_magnet       5    0.00017   0.00023  0.60000  260922-143515, 260922-143535, 260922-143557
==================== Videos with distractor ====================
         video_id   cam   session                                 template label top1_pred top1_conf correct_top1       session_temp prob_107 prob_73 prob_104 prob_42 prob_19
46  260922-143515  cam1  magnet_2  Putting [something] next to [something]   107       107   0.98429         True  distractor_magnet  0.98429 0.00663  0.00260 0.00244 0.00119

Top 5 Probability Breakdown:
  prob_107 -> 

In [138]:
avg_prob_in_diff_cases(df, distractor='books', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books =========================
label 48: Piling [something] up
cams used: ['cam1']
       session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0          Baseline      40    0.00033   0.00079  0.70000  260922-134214, 260922-134301, 260922-134333
1            Normal       5    0.10632   0.01906  0.00000  260922-144210, 260922-144221, 260922-144233
2  distractor_books       5    0.00001   0.00002  0.60000  260922-144349, 260922-144425, 260922-144448
==================== Videos with distractor ====================
         video_id   cam  session                                 template label top1_pred top1_conf correct_top1      session_temp prob_107 prob_114 prob_42 prob_103 prob_19
76  260922-144349  cam1  books_2  Putting [something] next to [something]   107       107   0.93409         True  distractor_books  0.93409  0.04104 0.00736  0.00629 0.00446

Top 5 Probability Breakdown:
  prob_107 ->  0.93409  |  Putting 

### A closer look
##### at individual examples and try to spot the ones where the distractor seems to work.
##### ***Beware that sigmoid_logits are used as probability

In [126]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','top1_conf','correct_top1']]
print(df.shape)
df.head(2)

(150, 8)


,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.70513,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.77079,True


In [127]:
def sigmoid(x):
    return 1/(1+np.exp(-x))
    
d_sigmoid_logits = sigmoid(d['logits'])
d_sigmoid_logits.shape

(150, 174)

In [128]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
# probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)
probs_df = pd.DataFrame(d_sigmoid_logits, columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,prob_0,prob_1,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.705132,True,0.106691,0.999704,...,0.460774,0.125065,0.370225,0.171061,0.214691,0.597843,0.924142,0.004070,0.000536,0.103749
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.770786,True,0.205615,0.999722,...,0.496765,0.189521,0.320821,0.169956,0.240775,0.337204,0.933431,0.005555,0.000688,0.174410


In [123]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("=" * 20, 'Videos with distractor', "=" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        print(row[meta_cols + top5_cols].to_frame().T)
        print('-'*10)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114 prob_107 prob_42  prob_6 prob_105
16  260922-141236  cam1  plug_2  Putting [something] next to [something]   107       114   0.70823        False  distractor_plug  0.99975  0.99887 0.99265 0.98999  0.98795
----------
         video_id   cam session                

In [124]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("-" * 25, distractor, "-" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("-" * 20, 'Videos with distractor', "-" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        
        # Rename top 5 prob columns to include full class label name
        rename_map = {}
        for col in top5_cols:
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), col))
            rename_map[col] = f"{col}: {class_name}"

        row_df = row[meta_cols + top5_cols].to_frame().T.rename(columns=rename_map)
        print(row_df)
        print("-" * 50)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

------------------------- plug -------------------------
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
-------------------- Videos with distractor --------------------
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114: Putting [something similar to other things that are already on the table] prob_107: Putting [something] next to [something] prob_42: Moving [something] closer to [something] prob_6: Covering [something] with [something] prob_105: Putting [something] in front of [